In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ydata_profiling

from eda_utils.csv_loader import read_csv
from eda_utils.globals import *
from eda_utils.plotter import plot_partial_result, plot_title_page
from matplotlib.backends.backend_pdf import PdfPages

In [ ]:
# IN_ROOT_DIR = pathlib.Path("ns-data-sources/spreading_potentials/multi_layer_networks/small_real").glob("*.csv")
# ANALYSED_NETWORK = 'toy_network'
# ANALYSED_NETWORK = 'lazega'
# ANALYSED_NETWORK = 'eu_transport_klm'
# ANALYSED_NETWORK = 'ckm_physicians'
# ANALYSED_NETWORK = 'eu_transportation'
# ANALYSED_NETWORK = 'aucs'

IN_ROOT_DIR = pathlib.Path("ns-data-sources/spreading_potentials/multi_layer_networks/arxiv_netscience_coauthorship").glob("**/*.csv")
# ANALYSED_NETWORK = "arxiv_netscience_coauthorship_math.oc"
ANALYSED_NETWORK = "arxiv_netscience_coauthorship"

OUT_ROOT_DIR = "eda"

In [ ]:
raw_csvs = []
for idx, file_path in enumerate(IN_ROOT_DIR):
    print(f"processing {idx}th file: {file_path.name}")
    raw_csvs.append(read_csv(file_path))

raw_csv = pd.concat(raw_csvs, axis=0, ignore_index=True)
print(f"\n\navailable networks: {raw_csv['network'].unique()}\n\n")
raw_csv.head()

In [ ]:
result_grouped = raw_csv.loc[raw_csv[NETWORK] == ANALYSED_NETWORK].groupby(
    by=[NETWORK, PROTOCOL, P, ACTOR]
)
result_mean = result_grouped.mean()
result_std = result_grouped.std()

result_mean.head()

In [ ]:
out_dir = pathlib.Path(f"{OUT_ROOT_DIR}/{ANALYSED_NETWORK}")
out_dir.mkdir(exist_ok=True, parents=True)

In [ ]:
ydata_profiling.ProfileReport(result_mean, title=f"{ANALYSED_NETWORK} mean").to_file(f"{out_dir}/mean.html")
ydata_profiling.ProfileReport(result_std, title=f"{ANALYSED_NETWORK} std").to_file(f"{out_dir}/std.html")

In [ ]:
result_all = pd.merge(result_mean, result_std, left_index=True, right_index=True, suffixes=("_avg", "_std")).sort_index().reset_index(ACTOR)
result_all.head()

In [ ]:
simulated_cases = {idx_name: list(idx_lvl) for idx_name, idx_lvl in zip(result_all.index.names, result_all.index.levels)}

pdf = PdfPages(out_dir / (f"plots.pdf"))

for network in simulated_cases[NETWORK]:

    # title page - network
    partial_fig = plot_title_page(network)
    partial_fig.savefig(pdf, format="pdf")
    plt.close(partial_fig)

    # obtain y axis values' range for this network
    maxx = result_all.loc[network].max()
    ex_range = [0, np.ceil(maxx[f"{EXPOSED}_avg"]).astype(int)]
    sl_range = [0, np.ceil(maxx[f"{SIMULATION_LENGTH}_avg"]).astype(int)]
    pi_range = [0, np.ceil(maxx[f"{PEAK_INFECTED}_avg"]).astype(int)]
    pt_range = [0, np.ceil(maxx[f"{PEAK_ITERATION}_avg"]).astype(int)]

    # mean result across the network
    partial_fig = plot_partial_result(
        partial_result=result_all.loc[network].groupby(ACTOR).mean().reset_index(),
        suptitle=r"$\bf{" + f"net-{network}" + "}$",
        sl_range=sl_range,
        ex_range=ex_range,
        pi_range=pi_range,
        pt_range=pt_range,
    )
    partial_fig.savefig(pdf, format="pdf")
    plt.close(partial_fig)

    for proto in simulated_cases[PROTOCOL]:

        # title page - protocol
        partial_fig = plot_title_page(proto)
        partial_fig.savefig(pdf, format="pdf")
        plt.close(partial_fig)

        # obtain y axis values' range for this network and protocol
        maxx = result_all.loc[network].loc[proto].max()
        ex_range = [0, np.ceil(maxx[f"{EXPOSED}_avg"]).astype(int)]
        sl_range = [0, np.ceil(maxx[f"{SIMULATION_LENGTH}_avg"]).astype(int)]
        pi_range = [0, np.ceil(maxx[f"{PEAK_INFECTED}_avg"]).astype(int)]
        pt_range = [0, np.ceil(maxx[f"{PEAK_ITERATION}_avg"]).astype(int)]

        # mean result across the network and protocol
        partial_fig = plot_partial_result(
            partial_result=result_all.loc[network].loc[proto].groupby(ACTOR).mean().reset_index(),
            suptitle=f"net-{network}--" + r"$\bf{" + f"proto-{proto}" + "}$",
            sl_range=sl_range,
            ex_range=ex_range,
            pi_range=pi_range,
            pt_range=pt_range,
        )
        partial_fig.savefig(pdf, format="pdf")
        plt.close(partial_fig)

        # plot result for all particular experiments
        for p in simulated_cases[P]:
            partial_fig = plot_partial_result(
                partial_result=result_all.loc[network].loc[proto].loc[p],
                suptitle=f"net-{network}--proto-{proto}--" + r"$\bf{" + f"p-{p}" + "}$",
                sl_range=sl_range,
                ex_range=ex_range,
                pi_range=pi_range,
                pt_range=pt_range,
            )
            partial_fig.savefig(pdf, format="pdf")
            plt.close(partial_fig)

pdf.close()

## Similarity of top actors across various spreading regimes

In [ ]:
simulated_cases = {idx_name: list(idx_lvl) for idx_name, idx_lvl in zip(result_all.index.names, result_all.index.levels)}
rankings_raw = {}

for proto in simulated_cases[PROTOCOL]:
    _rankings_raw = []
    for p in simulated_cases[P]:
        _rankings_raw.append(
            list(
                result_all.loc[ANALYSED_NETWORK].loc[proto].loc[p].sort_values(
                    f"{SIMULATION_LENGTH}_avg", ascending=False
                )[ACTOR]
            )
        )
    rankings_raw[proto] = np.array(_rankings_raw)

# rankings_raw

### Kendall's W (doesn't work yet)

In [ ]:
# a = np.array(
#     [
#         [5, 6, 9, 1, 3, 0, 8, 2, 4, 7, 10],
#         [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
#         [10, 9, 8, 7, 6, 4, 5, 3, 2, 1, 0],
#     ]
# )
# def prepare_R_matrix(actors_ranking):
#     R = np.zeros_like(actors_ranking)
#     for rankind_idx, ranking in enumerate(actors_ranking):
#         for element_pos, element_id in enumerate(ranking, 1):
#             R[rankind_idx, element_id] = element_pos
    
#     return R

# # prepare_R_matrix(a), a
# rankings_raw[:3, :]

In [ ]:
# from typing import Any


# def map_actor_names(raw_rankings: np.ndarray, actor_names: np.array) -> tuple[np.ndarray, dict[Any, int]]:
#     mapping_dict = {actor_name.item(): idx for idx, actor_name in enumerate(actor_names)}
#     mapped_ranking = np.ones_like(rankings_raw) * (rankings_raw.shape[1] - 1)
#     for rr_idx, raw_ranking in enumerate(raw_rankings):
#         for actor_idx, actor_name in enumerate(raw_ranking):
#             mapped_ranking[rr_idx, actor_idx] = mapping_dict[actor_name]
#     return mapped_ranking, mapping_dict


# def prepare_R_matrix(actors_ranking):
#     R = np.zeros_like(actors_ranking)
#     for rankind_idx, ranking in enumerate(actors_ranking):
#         for element_pos, element_id in enumerate(ranking, 1):
#             R[rankind_idx, element_id] = element_pos
#     return R


# def kendalls_W(R_matrix: np.ndarray) -> float:
#     m, n = R_matrix.shape
#     R_total = np.sum(R_matrix, axis=0)
#     R_dash = np.mean(R_total)
#     S = np.sum((R_total - R_dash) ** 2)
#     W = 12 * S / (m**2 * (n**3 - n))
#     return W

In [ ]:
# cut_w = []

# for cut in range(1, rankings_raw.shape[1] + 1):
#     cut_rankings = rankings_raw[:, :cut]
#     mapped_ranking, _ = map_actor_names(cut_rankings, np.unique(cut_rankings))
#     R_matrix = prepare_R_matrix(mapped_ranking)
#     W = kendalls_W(R_matrix)
#     cut_w.append(W)

# print(cut_w)


In [ ]:
# plt.scatter(x=np.arange(0, len(cut_w)), y=cut_w)

## My metric

In [ ]:
cuts = {}

for proto, ranking_raw in rankings_raw.items():

    increments = []
    num_uniques = []

    i_actors = set()
    for j in range(1, ranking_raw.shape[1]):
        j_sample = ranking_raw[:, :j]
        j_actors = set(np.unique(j_sample).tolist())
        increments.append(len(j_actors.difference(i_actors)))
        num_uniques.append(len(j_actors))
        i_actors = j_actors

    num_uniques = 100 * np.array(num_uniques) / len(np.unique(ranking_raw))
    increments = 100 * np.array(increments) / len(np.unique(ranking_raw))

    cuts[proto] = {"num_uniques": num_uniques, "increments": increments}


In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(9, 4.5))

for ax, proto in zip(axs, cuts):
    ax_1A = ax
    increments = cuts[proto]["increments"]
    num_uniques = cuts[proto]["num_uniques"]

    color = "tab:red"
    ax_1A.set_xlabel("top-k actors")
    ax_1A.set_ylabel("increment of unique actors between cuts", color=color)
    ax_1A.plot(np.arange(len(increments)), increments, color=color)
    ax_1A.tick_params(axis="y", labelcolor=color)

    ax_1B = ax_1A.twinx()

    color = "tab:blue"
    ax_1B.set_ylabel("% of all actors in the cut", color=color) 
    ax_1B.plot(np.arange(len(num_uniques)), num_uniques, color=color)
    ax_1B.tick_params(axis="y", labelcolor=color)
    ax_1B.set_title(f"protocol: {proto}")

fig.suptitle("Share of the same top-k actors across all spreading regimes")
fig.tight_layout()
plt.savefig(out_dir / (f"ranking.pdf"))
plt.show()